In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기: 2-5 train_test_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '3_train_clean3.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '3_val_clean3.parquet')

print(train.shape, val.shape)

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드


# ============================================================
# 0. optuna study파일 db(cat_optuna) 접근, study파일 불러오기
# ============================================================

# import optuna

# study = optuna.load_study(
#     study_name='catboost_2_clean',
#     storage=f"sqlite:///{PROJECT_ROOT / 'models' / 'cat_optuna.db'}"
# )

# print('현재까지 전체 trial 수: ', len(study.trials))
# print('현재까지 최적 trial: ', study.best_trial.number)
# print('현재까지 최적 RMSE: ', study.best_value)
# print('현재까지 최적 hp조합: ', study.best_params)

(616293, 35) (133316, 35)


In [2]:
# ============================================================
# 0. 라이브러리 불러오기
# ============================================================

import numpy as np
import optuna

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

c:\Users\soohok402\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [6]:
# =============================================================
# 2. train 데이터를 튜닝 검증용으로 재분할(val은 최종 val로 남겨둠)
# =============================================================
# 2-1. train -> train_optuna, val_optuna

# 기존 train의 event_id 확인
print(f"train의 event_id 개수: {train['event_id'].nunique()}")  # 98개

optuna_split_idx = int((train['event_id'].nunique()) * 0.8)
print(f"재분할 train_optuna event_id 개수: {optuna_split_idx}")
print()

train_optuna = train.loc[
    train['event_id'] <= optuna_split_idx
]
val_optuna = train.loc[
    train['event_id'] > optuna_split_idx
]

# 1. event_id 연속성 확인
print('event_id 최솟값:', train['event_id'].min())
print('event_id 최댓값:', train['event_id'].max())
print('event_id 고유 개수:', train['event_id'].nunique())

# 2. 실제 분할 결과 확인
print('train_optuna 사상 수:', train_optuna['event_id'].nunique())
print('val_optuna 사상 수:', val_optuna['event_id'].nunique())

# 3. 두 데이터셋의 event_id 중복 확인
print('중복 사상 수:', len(
    set(train_optuna['event_id']) & set(val_optuna['event_id'])
))

train의 event_id 개수: 98
재분할 train_optuna event_id 개수: 78

event_id 최솟값: 1
event_id 최댓값: 98
event_id 고유 개수: 98
train_optuna 사상 수: 78
val_optuna 사상 수: 20
중복 사상 수: 0


In [7]:
# ============================================================
# 1. 데이터 불러오기 및 Tidal / Normal 분리
# ============================================================

# 1-2. 조석 주기성 후보 관측소 9개
tidal_stations = [
    'ST0110', 'ST0326', 'ST0159',
    'ST0358', 'ST0347', 'ST0415',
    'ST0292', 'ST0313', 'ST0234'
]

# 1-3. Train_optuna 분리
tidal_train_optuna = train_optuna.loc[
    train_optuna['station_id'].isin(tidal_stations)
].copy()

normal_train_optuna = train_optuna.loc[
    ~train_optuna['station_id'].isin(tidal_stations)
].copy()

# 1-4. Val_optuna 분리
tidal_val_optuna = val_optuna.loc[
    val_optuna['station_id'].isin(tidal_stations)
].copy()

normal_val_optuna = val_optuna.loc[
    ~val_optuna['station_id'].isin(tidal_stations)
].copy()

# 1-5. 최종 Validation 분리
tidal_val = val.loc[
    val['station_id'].isin(tidal_stations)
].copy()

normal_val = val.loc[
    ~val['station_id'].isin(tidal_stations)
].copy()

# 1-6. 데이터 분리 검증
assert len(tidal_train_optuna) + len(normal_train_optuna) == len(train_optuna)
assert len(tidal_val_optuna) + len(normal_val_optuna) == len(val_optuna)
assert len(tidal_val) + len(normal_val) == len(val)

print(f'Train_optuna: {len(train_optuna):,}')
print(f'  Tidal: {len(tidal_train_optuna):,}')
print(f'  Normal: {len(normal_train_optuna):,}')

print(f'\nVal_optuna: {len(val_optuna):,}')
print(f'  Tidal: {len(tidal_val_optuna):,}')
print(f'  Normal: {len(normal_val_optuna):,}')

print(f'\nValidation: {len(val):,}')
print(f'  Tidal: {len(tidal_val):,}')
print(f'  Normal: {len(normal_val):,}')

Train_optuna: 403,984
  Tidal: 7,594
  Normal: 396,390

Val_optuna: 212,309
  Tidal: 4,091
  Normal: 208,218

Validation: 133,316
  Tidal: 2,566
  Normal: 130,750


In [8]:
# ============================================================
# 2. Tidal / Normal 독립변수 및 종속변수 분리
# ============================================================

# 2-1. Train_optuna - Tidal
X_tidal_train_optuna = tidal_train_optuna[num_cols_fe + ['station_id']].copy()
y_tidal_train_optuna = tidal_train_optuna['target_maxrise_6h']

# 2-2. Train_optuna - Normal
X_normal_train_optuna = normal_train_optuna[num_cols_fe + ['station_id']].copy()
y_normal_train_optuna = normal_train_optuna['target_maxrise_6h']

# 2-3. Val_optuna - Tidal
X_tidal_val_optuna = tidal_val_optuna[num_cols_fe + ['station_id']].copy()
y_tidal_val_optuna = tidal_val_optuna['target_maxrise_6h']

# 2-4. Val_optuna - Normal
X_normal_val_optuna = normal_val_optuna[num_cols_fe + ['station_id']].copy()
y_normal_val_optuna = normal_val_optuna['target_maxrise_6h']

# 2-5. 최종 Validation - Tidal
X_tidal_val = tidal_val[num_cols_fe + ['station_id']].copy()
y_tidal_val = tidal_val['target_maxrise_6h']

# 2-6. 최종 Validation - Normal
X_normal_val = normal_val[num_cols_fe + ['station_id']].copy()
y_normal_val = normal_val['target_maxrise_6h']

# 2-7. 독립변수 및 종속변수 정합성 검증
for X, y in [
    (X_tidal_train_optuna, y_tidal_train_optuna),
    (X_normal_train_optuna, y_normal_train_optuna),
    (X_tidal_val_optuna, y_tidal_val_optuna),
    (X_normal_val_optuna, y_normal_val_optuna),
    (X_tidal_val, y_tidal_val),
    (X_normal_val, y_normal_val)
]:
    assert X.index.equals(y.index), "X와 y의 인덱스 불일치"

print("6개 데이터셋 X, y 정합성 검증 완료")

6개 데이터셋 X, y 정합성 검증 완료


## Optuna 준비

## Tidal 데이터셋 CatBoost hp tune

In [ ]:
# 2-7. CatBoost 범주형 변수 처리
X_tidal_train_optuna['station_id'] = X_tidal_train_optuna['station_id'].astype(str)
X_normal_train_optuna['station_id'] = X_normal_train_optuna['station_id'].astype(str)

X_tidal_val_optuna['station_id'] = X_tidal_val_optuna['station_id'].astype(str)
X_normal_val_optuna['station_id'] = X_normal_val_optuna['station_id'].astype(str)

X_tidal_val['station_id'] = X_tidal_val['station_id'].astype(str)
X_normal_val['station_id'] = X_normal_val['station_id'].astype(str)

In [11]:
# ============================================================
# 3. Tidal CatBoost - Optuna objective 함수
# ============================================================

def objective_tidal(trial):

    # 1. 하이퍼파라미터 탐색
    depth = trial.suggest_int('depth', 4, 9)

    learning_rate = trial.suggest_float(
        'learning_rate', 0.01, 0.1, log=True
    )

    l2_leaf_reg = trial.suggest_float(
        'l2_leaf_reg', 1.0, 10.0, log=True
    )

    random_strength = trial.suggest_float(
        'random_strength', 0.01, 10.0, log=True
    )

    # 2. 모델 정의
    model = CatBoostRegressor(
        iterations=3000,
        depth=depth,
        learning_rate=learning_rate,
        l2_leaf_reg=l2_leaf_reg,
        random_strength=random_strength,
        loss_function='RMSE',
        random_seed=42,
        thread_count=2,
        verbose=False,
        allow_writing_files=False
    )

    # 3. 모델 학습
    model.fit(
        X_tidal_train_optuna,
        y_tidal_train_optuna,
        cat_features=['station_id'],
        eval_set=(X_tidal_val_optuna, y_tidal_val_optuna),
        early_stopping_rounds=100,
        verbose=False
    )

    # 4. 검증 데이터 예측
    pred = model.predict(X_tidal_val_optuna)

    # 5. RMSE 계산
    rmse = root_mean_squared_error(
        y_tidal_val_optuna,
        pred
    )

    return rmse


# ============================================================
# 4. Tidal CatBoost - Optuna 50 Trial 실행 및 DB 저장
# ============================================================

# 1. DB 저장 경로
db_path = PROJECT_ROOT / 'optuna_tidal_catboost.db'

# 2. Study 생성 및 기존 결과 불러오기
study_tidal = optuna.create_study(
    study_name='tidal_catboost1',
    storage=f"sqlite:///{PROJECT_ROOT / 'models' / 'optuna_tidal_catboost.db'}",
    load_if_exists=True,
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

# 3. Optuna 실행
study_tidal.optimize(
    objective_tidal,
    n_trials=50
)

# 4. 최적 결과 출력
print(f'최적 Trial: {study_tidal.best_trial.number}')
print(f'최적 RMSE: {study_tidal.best_value:.6f}')
print(f'최적 HP: {study_tidal.best_params}')

[I 2026-11-03 08:30:06,995] A new study created in RDB with name: tidal_catboost1
[I 2026-11-03 08:30:19,991] Trial 0 finished with value: 0.4680874445813262 and parameters: {'depth': 6, 'learning_rate': 0.08927180304353628, 'l2_leaf_reg': 5.395030966670228, 'random_strength': 0.6251373574521749}. Best is trial 0 with value: 0.4680874445813262.
[I 2026-11-03 08:31:00,755] Trial 1 finished with value: 0.47459921146850464 and parameters: {'depth': 4, 'learning_rate': 0.01432169828911152, 'l2_leaf_reg': 1.1430983876313219, 'random_strength': 3.9676050770529883}. Best is trial 0 with value: 0.4680874445813262.
[I 2026-11-03 08:31:34,903] Trial 2 finished with value: 0.45080358133484144 and parameters: {'depth': 7, 'learning_rate': 0.051059032093947576, 'l2_leaf_reg': 1.0485387725194617, 'random_strength': 8.123245085588687}. Best is trial 2 with value: 0.45080358133484144.
[I 2026-11-03 08:33:48,660] Trial 3 finished with value: 0.44613933643405584 and parameters: {'depth': 8, 'learning_ra

KeyboardInterrupt: 

In [12]:
print(study_tidal.best_params)

{'depth': 9, 'learning_rate': 0.09346240711186694, 'l2_leaf_reg': 1.1064945250951146, 'random_strength': 0.010397138264437182}


In [13]:
# ============================================================
# 5. Tidal CatBoost - Optuna 2차 objective 함수
# ============================================================

def objective_tidal2(trial):

    # 1. 하이퍼파라미터 탐색
    depth = trial.suggest_int(
        'depth', 8, 9
    )

    learning_rate = trial.suggest_float(
        'learning_rate', 0.04, 0.15, log=True
    )

    l2_leaf_reg = trial.suggest_float(
        'l2_leaf_reg', 0.1, 3.0, log=True
    )

    random_strength = trial.suggest_float(
        'random_strength', 0.001, 0.1, log=True
    )

    # 2. 모델 정의
    model = CatBoostRegressor(
        iterations=3000,
        depth=depth,
        learning_rate=learning_rate,
        l2_leaf_reg=l2_leaf_reg,
        random_strength=random_strength,
        loss_function='RMSE',
        random_seed=42,
        thread_count=2,
        verbose=False,
        allow_writing_files=False
    )

    # 3. 모델 학습
    model.fit(
        X_tidal_train_optuna,
        y_tidal_train_optuna,
        cat_features=['station_id'],
        eval_set=(X_tidal_val_optuna, y_tidal_val_optuna),
        early_stopping_rounds=100,
        verbose=False
    )

    # 4. 검증 데이터 예측 및 RMSE
    pred = model.predict(X_tidal_val_optuna)

    rmse = root_mean_squared_error(
        y_tidal_val_optuna,
        pred
    )

    return rmse

In [14]:
# ============================================================
# 6. Tidal CatBoost - Optuna 2차 20 Trial 실행 및 DB 저장
# ============================================================

# 1. Study 생성 및 기존 결과 불러오기
study_tidal2 = optuna.create_study(
    study_name='tidal_catboost2',
    storage=f"sqlite:///{PROJECT_ROOT / 'models' / 'optuna_tidal_catboost.db'}",
    load_if_exists=True,
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

# 2. 기존 1차 Best HP를 첫 Trial로 등록
if len(study_tidal2.trials) == 0:
    study_tidal2.enqueue_trial({
        'depth': 9,
        'learning_rate': 0.09346240711186694,
        'l2_leaf_reg': 1.1064945250951146,
        'random_strength': 0.010397138264437182
    })

# 3. 남은 Trial 수 계산
finished_trials = sum(
    t.state in (
        optuna.trial.TrialState.COMPLETE,
        optuna.trial.TrialState.FAIL,
        optuna.trial.TrialState.PRUNED
    )
    for t in study_tidal2.trials
)

remaining_trials = max(0, 20 - finished_trials)

# 4. Optuna 실행
if remaining_trials > 0:
    study_tidal2.optimize(
        objective_tidal2,
        n_trials=remaining_trials
    )

# 5. 최적 결과 출력
print(f'완료 Trial: {sum(t.state == optuna.trial.TrialState.COMPLETE for t in study_tidal2.trials)}')
print(f'Best Trial: {study_tidal2.best_trial.number}')
print(f'Best RMSE: {study_tidal2.best_value:.6f}')
print(f'Best HP: {study_tidal2.best_params}')

[I 2026-11-03 09:06:07,404] A new study created in RDB with name: tidal_catboost2
[I 2026-11-03 09:06:49,342] Trial 0 finished with value: 0.4412337115723262 and parameters: {'depth': 9, 'learning_rate': 0.09346240711186694, 'l2_leaf_reg': 1.1064945250951146, 'random_strength': 0.010397138264437182}. Best is trial 0 with value: 0.4412337115723262.
[I 2026-11-03 09:07:20,798] Trial 1 finished with value: 0.4513951753434553 and parameters: {'depth': 8, 'learning_rate': 0.14053992902758938, 'l2_leaf_reg': 1.2057126287443767, 'random_strength': 0.015751320499779727}. Best is trial 0 with value: 0.4412337115723262.
[I 2026-11-03 09:08:06,149] Trial 2 finished with value: 0.44206596176394186 and parameters: {'depth': 8, 'learning_rate': 0.04915930378506837, 'l2_leaf_reg': 0.12184186502221767, 'random_strength': 0.05399484409787434}. Best is trial 0 with value: 0.4412337115723262.
[I 2026-11-03 09:08:43,959] Trial 3 finished with value: 0.44547426944941504 and parameters: {'depth': 9, 'learni

완료 Trial: 20
Best Trial: 6
Best RMSE: 0.437405
Best HP: {'depth': 9, 'learning_rate': 0.04809875432044185, 'l2_leaf_reg': 0.27010527749605473, 'random_strength': 0.005404103854647328}


## Tidal 데이터셋 ExtraTrees hp tune

In [15]:
# ============================================================
# 4. ET 모델을 위한 Tidal 데이터 OneHotEncoding
# ============================================================

from sklearn.preprocessing import OneHotEncoder

# 1. OneHotEncoder 정의 및 학습
ohe = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

ohe.fit(X_tidal_train_optuna[['station_id']])

# 2. Train / Validation 원핫인코딩
X_tidal_train_optuna_et = pd.concat([
    X_tidal_train_optuna[num_cols_fe],
    pd.DataFrame(
        ohe.transform(X_tidal_train_optuna[['station_id']]),
        columns=ohe.get_feature_names_out(),
        index=X_tidal_train_optuna.index
    )
], axis=1)

X_tidal_val_optuna_et = pd.concat([
    X_tidal_val_optuna[num_cols_fe],
    pd.DataFrame(
        ohe.transform(X_tidal_val_optuna[['station_id']]),
        columns=ohe.get_feature_names_out(),
        index=X_tidal_val_optuna.index
    )
], axis=1)

# 3. 데이터 정합성 검증
assert X_tidal_train_optuna_et.index.equals(y_tidal_train_optuna.index)
assert X_tidal_val_optuna_et.index.equals(y_tidal_val_optuna.index)
assert X_tidal_train_optuna_et.columns.equals(X_tidal_val_optuna_et.columns)

print(X_tidal_train_optuna.shape, X_tidal_train_optuna_et.shape)
print(X_tidal_val_optuna.shape, X_tidal_val_optuna_et.shape)

(7594, 32) (7594, 40)
(4091, 32) (4091, 40)


In [16]:
# ============================================================
# 5. Tidal ET - Optuna objective 함수
# ============================================================

from sklearn.ensemble import ExtraTreesRegressor

def objective_et(trial):

    # 1. 하이퍼파라미터 탐색
    max_depth = trial.suggest_categorical(
        'max_depth', [None, 10, 15, 20, 25, 30, 40]
    )

    min_samples_split = trial.suggest_int(
        'min_samples_split', 2, 20
    )

    min_samples_leaf = trial.suggest_int(
        'min_samples_leaf', 1, 10
    )

    max_features = trial.suggest_float(
        'max_features', 0.5, 1.0
    )

    # 2. 모델 정의
    model = ExtraTreesRegressor(
        n_estimators=100,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42,
        n_jobs=2
    )

    # 3. 모델 학습
    model.fit(
        X_tidal_train_optuna_et,
        y_tidal_train_optuna
    )

    # 4. 검증 데이터 예측
    pred = model.predict(
        X_tidal_val_optuna_et
    )

    # 5. RMSE 계산
    rmse = root_mean_squared_error(
        y_tidal_val_optuna,
        pred
    )

    return rmse

In [17]:
# ============================================================
# 6. Tidal ET - Optuna 30 Trial 실행 및 DB 저장
# ============================================================

# 1. Study 생성 및 기존 결과 불러오기
study_tidal_et = optuna.create_study(
    study_name='tidal_et1',
    storage=f"sqlite:///{PROJECT_ROOT / 'models' / 'optuna_tidal_et.db'}",
    load_if_exists=True,
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

# 2. ET 기본 HP를 첫 Trial로 등록
if len(study_tidal_et.trials) == 0:
    study_tidal_et.enqueue_trial({
        'max_depth': None,
        'min_samples_split': 2,
        'min_samples_leaf': 1,
        'max_features': 1.0
    })

# 3. 남은 Trial 계산
completed_trials = sum(
    t.state == optuna.trial.TrialState.COMPLETE
    for t in study_tidal_et.trials
)

remaining_trials = max(0, 30 - completed_trials)

# 4. Optuna 실행
if remaining_trials > 0:
    study_tidal_et.optimize(
        objective_et,
        n_trials=remaining_trials
    )

# 5. 최적 결과 출력
print(f'완료된 Trial: {sum(t.state == optuna.trial.TrialState.COMPLETE for t in study_tidal_et.trials)}')
print(f'최적 Trial: {study_tidal_et.best_trial.number}')
print(f'최적 RMSE: {study_tidal_et.best_value:.6f}')
print(f'최적 HP: {study_tidal_et.best_params}')

[I 2026-11-03 09:28:48,904] A new study created in RDB with name: tidal_et1
[I 2026-11-03 09:28:51,813] Trial 0 finished with value: 0.42102036902529666 and parameters: {'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 1.0}. Best is trial 0 with value: 0.42102036902529666.
[I 2026-11-03 09:28:52,749] Trial 1 finished with value: 0.45471825941341903 and parameters: {'max_depth': 10, 'min_samples_split': 18, 'min_samples_leaf': 7, 'max_features': 0.8540362888980227}. Best is trial 0 with value: 0.42102036902529666.
[I 2026-11-03 09:28:53,513] Trial 2 finished with value: 0.466562188444179 and parameters: {'max_depth': 10, 'min_samples_split': 11, 'min_samples_leaf': 5, 'max_features': 0.645614570099021}. Best is trial 0 with value: 0.42102036902529666.
[I 2026-11-03 09:28:54,384] Trial 3 finished with value: 0.43863501125330473 and parameters: {'max_depth': 30, 'min_samples_split': 11, 'min_samples_leaf': 6, 'max_features': 0.5232252063599989}. Best is tr

완료된 Trial: 30
최적 Trial: 22
최적 RMSE: 0.420152
최적 HP: {'max_depth': 40, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.731843786694916}


In [19]:
# ============================================================
# 7. Tidal ET - Optuna 2차 objective 함수
# ============================================================

def objective_et2(trial):

    # 1. 하이퍼파라미터 탐색
    n_estimators = trial.suggest_categorical(
        'n_estimators', [100, 200, 300]
    )

    max_depth = trial.suggest_categorical(
        'max_depth', [None, 20, 30, 40, 50]
    )

    max_features = trial.suggest_float(
        'max_features', 0.6, 1.0
    )

    # 2. 모델 정의
    model = ExtraTreesRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features=max_features,
        random_state=42,
        n_jobs=2
    )

    # 3. 모델 학습
    model.fit(
        X_tidal_train_optuna_et,
        y_tidal_train_optuna
    )

    # 4. 검증 데이터 예측
    pred = model.predict(X_tidal_val_optuna_et)

    # 5. RMSE 계산
    rmse = root_mean_squared_error(
        y_tidal_val_optuna,
        pred
    )

    return rmse

In [20]:
# ============================================================
# 8. Tidal ET - Optuna 2차 30 Trial 실행 및 DB 저장
# ============================================================

# 1. Study 생성 및 기존 결과 불러오기
study_tidal_et2 = optuna.create_study(
    study_name='tidal_et2',
    storage=f"sqlite:///{PROJECT_ROOT / 'models' / 'optuna_tidal_et.db'}",
    load_if_exists=True,
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

# 2. 기존 1차 Best HP를 첫 Trial로 등록
if len(study_tidal_et2.trials) == 0:
    study_tidal_et2.enqueue_trial({
        'n_estimators': 100,
        'max_depth': 40,
        'max_features': 0.731843786694916
    })

# 3. 남은 Trial 계산
finished_trials = sum(
    t.state in (
        optuna.trial.TrialState.COMPLETE,
        optuna.trial.TrialState.FAIL,
        optuna.trial.TrialState.PRUNED
    )
    for t in study_tidal_et2.trials
)

remaining_trials = max(0, 30 - finished_trials)

# 4. Optuna 실행
if remaining_trials > 0:
    study_tidal_et2.optimize(
        objective_et2,
        n_trials=remaining_trials
    )

# 5. 최적 결과 출력
print(f'완료 Trial: {sum(t.state == optuna.trial.TrialState.COMPLETE for t in study_tidal_et2.trials)}')
print(f'Best Trial: {study_tidal_et2.best_trial.number}')
print(f'Best RMSE: {study_tidal_et2.best_value:.6f}')
print(f'Best HP: {study_tidal_et2.best_params}')

[I 2026-11-03 09:31:35,424] Using an existing study with `study_name='tidal_et2'` instead of creating a new one.
[I 2026-11-03 09:31:37,557] Trial 0 finished with value: 0.4201519991917627 and parameters: {'n_estimators': 100, 'max_depth': 40, 'max_features': 0.731843786694916}. Best is trial 0 with value: 0.4201519991917627.
[I 2026-11-03 09:31:42,211] Trial 1 finished with value: 0.4186223598162287 and parameters: {'n_estimators': 200, 'max_depth': 50, 'max_features': 0.8404460046972835}. Best is trial 1 with value: 0.4186223598162287.
[I 2026-11-03 09:31:48,892] Trial 2 finished with value: 0.41881406817830974 and parameters: {'n_estimators': 300, 'max_depth': None, 'max_features': 0.8099025726528951}. Best is trial 1 with value: 0.4186223598162287.
[I 2026-11-03 09:31:54,633] Trial 3 finished with value: 0.41851121383585177 and parameters: {'n_estimators': 300, 'max_depth': 50, 'max_features': 0.6798695128633439}. Best is trial 3 with value: 0.41851121383585177.
[I 2026-11-03 09:31

완료 Trial: 30
Best Trial: 24
Best RMSE: 0.418008
Best HP: {'n_estimators': 200, 'max_depth': None, 'max_features': 0.7843011071485029}


In [ ]:
print(study.best_params)

In [ ]:
# ============================================================
# 6. iteration 3000으로 단일모델 한번 돌려보기(optuna셋으로)
# ============================================================

model_iter_3000 = CatBoostRegressor(
    **study.best_params,
    iterations=3000,  # 최대 3000회 부스팅까지 허용
    loss_function='RMSE',
    random_seed=42,  # random_state와 동일한 효과
    thread_count=2,  # RF의 n_jobs와 동일: 병렬작업 사용개수
    verbose=200,  # 200개 트리마다 현황정보 표기
    allow_writing_files=False  # 학습 중 로그 등을 별도로 기록, 분석과 무관하므로 불필요
)

# 3. 모델 학습
model_iter_3000.fit(
    X_train_optuna_ohe,
    y_train_optuna,
    eval_set=(X_val_optuna_ohe, y_val_optuna),  # 내부 검증셋
    early_stopping_rounds=100  # RMSE가 100번 연속 개선되지 않으면 종료
)

# 4. 검증 데이터 예측
pred_iter_3000 = model_iter_3000.predict(
    X_val_optuna_ohe
)

# 5. RMSE 계산
rmse_iter_3000 = root_mean_squared_error(
    y_val_optuna,
    pred_iter_3000
)

print(f"Validation RMSE: {rmse_iter_3000:.6f}")
print(f"Best iteration: {model_iter_3000.get_best_iteration()}")

In [ ]:
# ============================================================
# 6. iteration 3000 대조군(나머지는 기본 조건으로)
# ============================================================

model_iter_3000 = CatBoostRegressor(
    iterations=3000,  # 최대 3000회 부스팅까지 허용
    loss_function='RMSE',
    random_seed=42,  # random_state와 동일한 효과
    thread_count=2,  # RF의 n_jobs와 동일: 병렬작업 사용개수
    verbose=200,  # 200개 트리마다 현황정보 표기
    allow_writing_files=False  # 학습 중 로그 등을 별도로 기록, 분석과 무관하므로 불필요
)

# 3. 모델 학습
model_iter_3000.fit(
    X_train_optuna_ohe,
    y_train_optuna,
    eval_set=(X_val_optuna_ohe, y_val_optuna),  # 내부 검증셋
    early_stopping_rounds=100  # RMSE가 100번 연속 개선되지 않으면 종료
)

# 4. 검증 데이터 예측
pred_iter_3000 = model_iter_3000.predict(
    X_val_optuna_ohe
)

# 5. RMSE 계산
rmse_iter_3000 = root_mean_squared_error(
    y_val_optuna,
    pred_iter_3000
)

print(f"Validation RMSE: {rmse_iter_3000:.6f}")
print(f"Best iteration: {model_iter_3000.get_best_iteration()}")